In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json, zipfile

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix
from baseline import calculate_resilience_cost
SEED = 42
np.random.seed(SEED)

# --- Load data ---
df = pd.read_csv("data/train.csv", index_col=0)
test_df = pd.read_csv("data/test.csv", index_col=0)
cost_matrix = pd.read_csv("data/cost_matrix.csv", index_col=0).values

display(df)

In [ ]:
# Data inspection
display(df.describe())


In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder ,OrdinalEncoder
categories=[["green", "yellow", "orange", "red"]]
# --- Encode target and split ---
enc = OrdinalEncoder(categories=categories, dtype=int)
display(enc)
enc.fit(df.loc[:, ["alert"]])
display(enc.categories_)
y = enc.transform(df.loc[:, ["alert"]]).ravel()
X = df.drop("alert", axis=1)

display(X.corr())
display(df.info())

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

pipe = make_pipeline(StandardScaler())
numeric_features = ["magnitude", "depth", "cdi", "mmi", "sig"]
numeric_transformer = Pipeline(
    # steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    steps=[("scaler", StandardScaler())]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        # ("cat", categorical_transformer, categorical_features),
    ]
)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.compose import TransformedTargetRegressor

clf = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", KNeighborsClassifier(n_neighbors=10))]
)

clf

In [ ]:

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

clf.fit(X_train, y_train )

In [ ]:
# # --- Scale features ---
# scaler = MinMaxScaler().fit(X_train)
# X_train, X_val, X_test = (
#     scaler.transform(X_train),
#     scaler.transform(X_val),
#     scaler.transform(test_df),
# )

In [ ]:

# # # --- Train baseline model ---
# clf = KNeighborsClassifier(n_neighbors=10)
# clf.fit(X_train, y_train)
# clf


In [ ]:


# --- Validate ---
y_val_pred = clf.predict(X_val)
f1 = f1_score(y_val, y_val_pred, average="macro")
cm = confusion_matrix(enc.inverse_transform(y_val.reshape(-1, 1)), enc.inverse_transform(y_val_pred.reshape(-1, 1)))
rci = calculate_resilience_cost(cm, cost_matrix)

print(f"F1 (macro): {f1:.3f}")
print("Confusion matrix:\n", cm)
print(f"Resilience Cost: {rci:.2f}")